# ERGT Colab Phase 3 Confirm Seed

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

This confirmation run keeps the short proof budget fixed and changes the seed to `2027`. It tests whether `real_d alpha=0.2` remains stronger than the same-seed baseline and random distance controls.

In [ ]:
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"

DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
DEVICE = "cuda"

BASELINE_CONFIG = "configs/baseline/confirm_seed_wikitext2.json"
BASELINE_RESULT = "runs/phase0_baseline/confirm_seed_wikitext2/baseline_results.json"

CONFIRM_CONFIGS = {
    "real_d_alpha_0_2": "configs/ergt_v1/confirm_seed/real_d_alpha_0_2.json",
    "random_d_alpha_0_2": "configs/ergt_v1/confirm_seed/random_d_alpha_0_2.json",
    "random_d_alpha_0_1": "configs/ergt_v1/confirm_seed/random_d_alpha_0_1.json",
}

CONFIRM_RESULTS = {
    "real_d_alpha_0_2": "runs/phase3_geo_attention/confirm_seed/real_d_alpha_0_2/metrics.json",
    "random_d_alpha_0_2": "runs/phase3_geo_attention/confirm_seed/random_d_alpha_0_2/metrics.json",
    "random_d_alpha_0_1": "runs/phase3_geo_attention/confirm_seed/random_d_alpha_0_1/metrics.json",
}

COMPARISON_DIR = "runs/phase3_geo_attention/confirm_seed"
COMPARISON_RESULT = f"{COMPARISON_DIR}/confirm_seed_results.json"

RUN_BASELINE = True
RUN_ERGT = True
RUN_COMPARISON = True
FORCE_RERUN = False

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Set Colab hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

## 2. Clone/update repo and install dependencies

In [ ]:
project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
)
print("repo:", repo_head.strip())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,data,viz]"],
    check=True,
)

## 3. Prepare WikiText-2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Run same-seed baseline and ERGT controls

In [ ]:
def run_command(command):
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


baseline_path = Path(PROJECT_DIR) / BASELINE_RESULT
if RUN_BASELINE:
    if baseline_path.exists() and not FORCE_RERUN:
        print("Skipping existing baseline:", baseline_path)
    else:
        run_command(
            [
                sys.executable,
                "experiments/train_baseline.py",
                "--config",
                BASELINE_CONFIG,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("RUN_BASELINE is False; no baseline run executed.")

if not baseline_path.exists():
    raise FileNotFoundError(f"Missing baseline result: {BASELINE_RESULT}")

if RUN_ERGT:
    for label, config_path in CONFIRM_CONFIGS.items():
        result_path = Path(PROJECT_DIR) / CONFIRM_RESULTS[label]
        if result_path.exists() and not FORCE_RERUN:
            print("Skipping existing ERGT run:", label, result_path)
            continue
        print("\n=== Confirm seed condition:", label, "===")
        run_command(
            [
                sys.executable,
                "experiments/train_ergt_v1.py",
                "--config",
                config_path,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("RUN_ERGT is False; no ERGT run executed.")

for label, path in CONFIRM_RESULTS.items():
    if not (Path(PROJECT_DIR) / path).exists():
        raise FileNotFoundError(f"Missing confirm result for {label}: {path}")

## 5. Compare confirm-seed results

In [ ]:
if RUN_COMPARISON:
    run_command(
        [
            sys.executable,
            "experiments/compare_confirm_seed.py",
            "--baseline",
            BASELINE_RESULT,
            "--real-d",
            CONFIRM_RESULTS["real_d_alpha_0_2"],
            "--random-d-matched",
            CONFIRM_RESULTS["random_d_alpha_0_2"],
            "--random-d-best-control",
            CONFIRM_RESULTS["random_d_alpha_0_1"],
            "--output-dir",
            COMPARISON_DIR,
        ]
    )
else:
    print("RUN_COMPARISON is False; no comparison generated.")

## 6. Print confirm-seed report

In [ ]:
report_path = Path(PROJECT_DIR) / COMPARISON_RESULT
with report_path.open("r", encoding="utf-8") as handle:
    report = json.load(handle)

print(json.dumps(report, indent=2, sort_keys=True)[:12000])

## 7. Archive light outputs

In [ ]:
light_root = Path("/content/ergt_confirm_seed_light")
if light_root.exists():
    shutil.rmtree(light_root)

include_roots = [
    "runs/phase0_baseline/confirm_seed_wikitext2",
    COMPARISON_DIR,
]

for include_root in include_roots:
    source_dir = Path(PROJECT_DIR) / include_root
    target_dir = light_root / include_root
    target_dir.mkdir(parents=True, exist_ok=True)
    for path in source_dir.rglob("*"):
        if not path.is_file() or path.suffix not in {".json", ".jsonl"}:
            continue
        relative = path.relative_to(source_dir)
        destination = target_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)
        print("included:", destination.relative_to(light_root))

light_archive = shutil.make_archive("/content/ergt_confirm_seed_light", "zip", light_root)
print("Light archive ready:", light_archive)